# Delay Comparison: Unlubricated vs Lubricated

Compare raw theta_trk and theta_field for both datasets to visualize delay differences.

In [9]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go

## Load Unlubricated Data

In [10]:
# Unlubricated configuration
TRACK_FILE_UNLUBE = 'chirp-data/unlube3-1.csv'
FIELD_FILE_UNLUBE = 'chirp3-unlube.csv'
B_START_UNLUBE = 1529
B_END_UNLUBE = 6521

# Load tracking data
track_tbl_unlube = pd.read_csv(TRACK_FILE_UNLUBE, header=1)
t_trk_unlube = track_tbl_unlube['t'].values
theta_trk_unlube = track_tbl_unlube['θ'].values / 180 * np.pi

# Handle NaN values
if np.isnan(theta_trk_unlube).sum() > 0:
    theta_series = pd.Series(theta_trk_unlube)
    theta_trk_unlube = theta_series.bfill().ffill().values

# Load field data
field_tbl_unlube = pd.read_csv(FIELD_FILE_UNLUBE, skiprows=6, sep=';')
idx_unlube = field_tbl_unlube['Index'].values
t_field_unlube = idx_unlube * 1e-3
t_field_unlube = t_field_unlube[B_START_UNLUBE:B_END_UNLUBE]
t_field_unlube = t_field_unlube - t_field_unlube[0]  # Start from zero

Bx_unlube = field_tbl_unlube['Bx[mT]'].values[B_START_UNLUBE:B_END_UNLUBE]
By_unlube = field_tbl_unlube['By[mT]'].values[B_START_UNLUBE:B_END_UNLUBE]
Bz_unlube = field_tbl_unlube['Bz[mT]'].values[B_START_UNLUBE:B_END_UNLUBE]

theta_field_wrapped_unlube = np.arctan2(Bz_unlube, By_unlube)
theta_field_unlube = np.unwrap(theta_field_wrapped_unlube)

print(f"Unlubricated - Tracking: {len(t_trk_unlube)} samples, [{t_trk_unlube[0]:.3f}, {t_trk_unlube[-1]:.3f}] s")
print(f"Unlubricated - Field: {len(t_field_unlube)} samples, [{t_field_unlube[0]:.3f}, {t_field_unlube[-1]:.3f}] s")

Unlubricated - Tracking: 10070 samples, [0.000, 5.035] s
Unlubricated - Field: 4992 samples, [0.000, 4.991] s


## Load Lubricated Data

In [11]:
# Lubricated configuration
TRACK_FILE_LUBE = 'chirp-data/lube3-1.csv'
FIELD_FILE_LUBE = 'chirp3-lube.csv'
B_START_LUBE = 212
B_END_LUBE = 5181

# Load tracking data
track_tbl_lube = pd.read_csv(TRACK_FILE_LUBE, header=1)
t_trk_lube = track_tbl_lube['t'].values
theta_trk_lube = track_tbl_lube['θ'].values / 180 * np.pi

# Handle NaN values
if np.isnan(theta_trk_lube).sum() > 0:
    theta_series = pd.Series(theta_trk_lube)
    theta_trk_lube = theta_series.bfill().ffill().values

# Load field data
field_tbl_lube = pd.read_csv(FIELD_FILE_LUBE, skiprows=6, sep=';')
idx_lube = field_tbl_lube['Index'].values
t_field_lube = idx_lube * 1e-3
t_field_lube = t_field_lube[B_START_LUBE:B_END_LUBE]
t_field_lube = t_field_lube - t_field_lube[0]  # Start from zero

Bx_lube = field_tbl_lube['Bx[mT]'].values[B_START_LUBE:B_END_LUBE]
By_lube = field_tbl_lube['By[mT]'].values[B_START_LUBE:B_END_LUBE]
Bz_lube = field_tbl_lube['Bz[mT]'].values[B_START_LUBE:B_END_LUBE]

theta_field_wrapped_lube = np.arctan2(Bz_lube, By_lube)
theta_field_lube = np.unwrap(theta_field_wrapped_lube)

print(f"Lubricated - Tracking: {len(t_trk_lube)} samples, [{t_trk_lube[0]:.3f}, {t_trk_lube[-1]:.3f}] s")
print(f"Lubricated - Field: {len(t_field_lube)} samples, [{t_field_lube[0]:.3f}, {t_field_lube[-1]:.3f}] s")

Lubricated - Tracking: 10070 samples, [0.000, 5.035] s
Lubricated - Field: 4969 samples, [0.000, 4.968] s


## Plot All 4 Curves Overlaid

In [12]:
# Plot unwrapped theta - all 4 curves
fig_unwrapped = go.Figure()

fig_unwrapped.add_trace(go.Scatter(
    x=t_trk_unlube, y=theta_trk_unlube, mode='lines',
    name='Unlubricated θ_trk',
    line=dict(color='blue', width=2),
    opacity=0.8
))

fig_unwrapped.add_trace(go.Scatter(
    x=t_field_unlube, y=theta_field_unlube, mode='lines',
    name='Unlubricated θ_field',
    line=dict(color='orange', width=2),
    opacity=0.8
))

fig_unwrapped.add_trace(go.Scatter(
    x=t_trk_lube, y=theta_trk_lube, mode='lines',
    name='Lubricated θ_trk',
    line=dict(color='green', width=2, dash='dash'),
    opacity=0.8
))

fig_unwrapped.add_trace(go.Scatter(
    x=t_field_lube, y=theta_field_lube, mode='lines',
    name='Lubricated θ_field',
    line=dict(color='red', width=2, dash='dash'),
    opacity=0.8
))

fig_unwrapped.update_layout(
    title='All 4 Curves Overlaid: Unwrapped Theta (NO INTERPOLATION)',
    xaxis_title='Time (s)',
    yaxis_title='Angle (rad)',
    hovermode='closest',
    width=1400,
    height=700,
    legend=dict(x=0.02, y=0.98)
)
fig_unwrapped.add_hline(y=0, line_dash="dash", line_color="gray", opacity=0.3)
fig_unwrapped.show()

In [13]:
# Plot wrapped theta - all 4 curves
theta_trk_wrapped_unlube = np.arctan2(np.sin(theta_trk_unlube), np.cos(theta_trk_unlube))
theta_trk_wrapped_lube = np.arctan2(np.sin(theta_trk_lube), np.cos(theta_trk_lube))

fig_wrapped = go.Figure()

fig_wrapped.add_trace(go.Scatter(
    x=t_trk_unlube, y=theta_trk_wrapped_unlube, mode='lines',
    name='Unlubricated θ_trk',
    line=dict(color='blue', width=2),
    opacity=0.8
))

fig_wrapped.add_trace(go.Scatter(
    x=t_field_unlube, y=theta_field_wrapped_unlube, mode='lines',
    name='Unlubricated θ_field',
    line=dict(color='orange', width=2),
    opacity=0.8
))

fig_wrapped.add_trace(go.Scatter(
    x=t_trk_lube, y=theta_trk_wrapped_lube, mode='lines',
    name='Lubricated θ_trk',
    line=dict(color='green', width=2, dash='dash'),
    opacity=0.8
))

fig_wrapped.add_trace(go.Scatter(
    x=t_field_lube, y=theta_field_wrapped_lube, mode='lines',
    name='Lubricated θ_field',
    line=dict(color='red', width=2, dash='dash'),
    opacity=0.8
))

fig_wrapped.update_layout(
    title='All 4 Curves Overlaid: Wrapped Theta (NO INTERPOLATION)',
    xaxis_title='Time (s)',
    yaxis_title='Angle (rad)',
    hovermode='closest',
    width=1400,
    height=700,
    legend=dict(x=0.02, y=0.98)
)
fig_wrapped.add_hline(y=0, line_dash="dash", line_color="gray", opacity=0.3)
fig_wrapped.add_hline(y=np.pi, line_dash="dash", line_color="red", opacity=0.3)
fig_wrapped.add_hline(y=-np.pi, line_dash="dash", line_color="red", opacity=0.3)
fig_wrapped.show()

## Summary Statistics

In [7]:
# Load uncut (full) lubricated field data
field_tbl_lube_full = pd.read_csv(FIELD_FILE_LUBE, skiprows=6, sep=';')
idx_lube_full = field_tbl_lube_full['Index'].values
t_field_lube_full = idx_lube_full * 1e-3  # Convert ms to seconds
t_field_lube_full = t_field_lube_full - t_field_lube_full[0]  # Start from zero

Bx_lube_full = field_tbl_lube_full['Bx[mT]'].values
By_lube_full = field_tbl_lube_full['By[mT]'].values
Bz_lube_full = field_tbl_lube_full['Bz[mT]'].values

theta_field_wrapped_lube_full = np.arctan2(Bz_lube_full, By_lube_full)
theta_field_lube_full = np.unwrap(theta_field_wrapped_lube_full)

print(f"Uncut lubricated field: {len(t_field_lube_full)} samples")
print(f"Time range: [{t_field_lube_full[0]:.3f}, {t_field_lube_full[-1]:.3f}] s")
print(f"Theta range: [{np.min(theta_field_lube_full):.2f}, {np.max(theta_field_lube_full):.2f}] rad")

# Plot uncut lubricated field (unwrapped)
fig_uncut_unwrapped = go.Figure()
fig_uncut_unwrapped.add_trace(go.Scatter(
    x=idx_lube_full, y=theta_field_lube_full, mode='lines',
    name='Uncut Lubricated θ_field (unwrapped)',
    line=dict(color='purple', width=2),
    opacity=0.8
))
fig_uncut_unwrapped.update_layout(
    title='Uncut Lubricated Field: Unwrapped Theta (Full Dataset)',
    xaxis_title='Index',
    yaxis_title='Angle (rad)',
    hovermode='closest',
    width=1400,
    height=600
)
fig_uncut_unwrapped.add_hline(y=0, line_dash="dash", line_color="gray", opacity=0.3)
fig_uncut_unwrapped.show()

# Plot uncut lubricated field (wrapped)
fig_uncut_wrapped = go.Figure()
fig_uncut_wrapped.add_trace(go.Scatter(
    x=t_field_lube_full, y=theta_field_wrapped_lube_full, mode='lines',
    name='Uncut Lubricated θ_field (wrapped)',
    line=dict(color='purple', width=2),
    opacity=0.8
))
fig_uncut_wrapped.update_layout(
    title='Uncut Lubricated Field: Wrapped Theta (Full Dataset)',
    xaxis_title='Time (s)',
    yaxis_title='Angle (rad)',
    hovermode='closest',
    width=1400,
    height=600
)
fig_uncut_wrapped.add_hline(y=0, line_dash="dash", line_color="gray", opacity=0.3)
fig_uncut_wrapped.add_hline(y=np.pi, line_dash="dash", line_color="red", opacity=0.3)
fig_uncut_wrapped.add_hline(y=-np.pi, line_dash="dash", line_color="red", opacity=0.3)
fig_uncut_wrapped.show()

Uncut lubricated field: 5855 samples
Time range: [0.000, 5.854] s
Theta range: [-95.32, 120.78] rad


In [8]:
# Load uncut (full) unlubricated field data
field_tbl_unlube_full = pd.read_csv(FIELD_FILE_UNLUBE, skiprows=6, sep=';')
idx_unlube_full = field_tbl_unlube_full['Index'].values
t_field_unlube_full = idx_unlube_full * 1e-3  # Convert ms to seconds
t_field_unlube_full = t_field_unlube_full - t_field_unlube_full[0]  # Start from zero

Bx_unlube_full = field_tbl_unlube_full['Bx[mT]'].values
By_unlube_full = field_tbl_unlube_full['By[mT]'].values
Bz_unlube_full = field_tbl_unlube_full['Bz[mT]'].values

theta_field_wrapped_unlube_full = np.arctan2(Bz_unlube_full, By_unlube_full)
theta_field_unlube_full = np.unwrap(theta_field_wrapped_unlube_full)

print(f"Uncut unlubricated field: {len(t_field_unlube_full)} samples")
print(f"Time range: [{t_field_unlube_full[0]:.3f}, {t_field_unlube_full[-1]:.3f}] s")
print(f"Theta range: [{np.min(theta_field_unlube_full):.2f}, {np.max(theta_field_unlube_full):.2f}] rad")

# Plot uncut unlubricated field (unwrapped)
fig_uncut_unlube_unwrapped = go.Figure()
fig_uncut_unlube_unwrapped.add_trace(go.Scatter(
    x=idx_unlube_full, y=theta_field_unlube_full, mode='lines',
    name='Uncut Unlubricated θ_field (unwrapped)',
    line=dict(color='brown', width=2),
    opacity=0.8
))
fig_uncut_unlube_unwrapped.update_layout(
    title='Uncut Unlubricated Field: Unwrapped Theta (Full Dataset)',
    xaxis_title='Index',
    yaxis_title='Angle (rad)',
    hovermode='closest',
    width=1400,
    height=600
)
fig_uncut_unlube_unwrapped.add_hline(y=0, line_dash="dash", line_color="gray", opacity=0.3)
fig_uncut_unlube_unwrapped.show()

# Plot uncut unlubricated field (wrapped)
fig_uncut_unlube_wrapped = go.Figure()
fig_uncut_unlube_wrapped.add_trace(go.Scatter(
    x=idx_unlube_full, y=theta_field_wrapped_unlube_full, mode='lines',
    name='Uncut Unlubricated θ_field (wrapped)',
    line=dict(color='brown', width=2),
    opacity=0.8
))
fig_uncut_unlube_wrapped.update_layout(
    title='Uncut Unlubricated Field: Wrapped Theta (Full Dataset)',
    xaxis_title='Index',
    yaxis_title='Angle (rad)',
    hovermode='closest',
    width=1400,
    height=600
)
fig_uncut_unlube_wrapped.add_hline(y=0, line_dash="dash", line_color="gray", opacity=0.3)
fig_uncut_unlube_wrapped.add_hline(y=np.pi, line_dash="dash", line_color="red", opacity=0.3)
fig_uncut_unlube_wrapped.add_hline(y=-np.pi, line_dash="dash", line_color="red", opacity=0.3)
fig_uncut_unlube_wrapped.show()

Uncut unlubricated field: 7088 samples
Time range: [0.000, 7.087] s
Theta range: [-76.74, 139.58] rad


In [6]:
print("="*70)
print("DELAY COMPARISON SUMMARY")
print("="*70)

print("\nUnlubricated Dataset:")
print(f"  Tracking time range: [{t_trk_unlube[0]:.3f}, {t_trk_unlube[-1]:.3f}] s")
print(f"  Field time range: [{t_field_unlube[0]:.3f}, {t_field_unlube[-1]:.3f}] s")
print(f"  Overlap: [{max(t_trk_unlube[0], t_field_unlube[0]):.3f}, {min(t_trk_unlube[-1], t_field_unlube[-1]):.3f}] s")

print("\nLubricated Dataset:")
print(f"  Tracking time range: [{t_trk_lube[0]:.3f}, {t_trk_lube[-1]:.3f}] s")
print(f"  Field time range: [{t_field_lube[0]:.3f}, {t_field_lube[-1]:.3f}] s")
print(f"  Overlap: [{max(t_trk_lube[0], t_field_lube[0]):.3f}, {min(t_trk_lube[-1], t_field_lube[-1]):.3f}] s")

print("\n" + "="*70)
print("Use the interactive plots above to zoom and compare delays visually.")
print("="*70)

DELAY COMPARISON SUMMARY

Unlubricated Dataset:
  Tracking time range: [0.000, 5.035] s
  Field time range: [0.000, 4.992] s
  Overlap: [0.000, 4.992] s

Lubricated Dataset:
  Tracking time range: [0.000, 5.035] s
  Field time range: [0.000, 4.966] s
  Overlap: [0.000, 4.966] s

Use the interactive plots above to zoom and compare delays visually.
